## Write Urself Challenge
this is a section where I have to rewrite everything from memory and figure it out myself

In [2]:
import math

In [32]:
class Value:
    def __init__(self, num, _child=(), label=''):
        self.data   = num
        self.label  = label

        self._prev  = set(_child)
        self._backward = lambda: None
        self.grad   = 0.0

    # print
    def __repr__(self):
        return f"Value({self.label}:{self.data})"


    # basic operations
    def __add__ (self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad +=  self.data * out.grad 
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad = other.data * (self.data**(other.data-1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1 

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    # activation func
    def sigmoid(self):
        neg = -1 * self


    # backprop
    def backward(self):
        topo=[]
        vis =set()

        def build_topo(val):
            if val not in vis:
                vis.add(val)
                for c in val._prev:     
                    build_topo(c)
                topo.append(val)

        build_topo(self)
        self.grad = 1.0
        for val in reversed(topo):
            val._backward()

In [33]:
z = Value(3.0, label='z')
y = Value(6.0, label='y')
y /z

Value(:2.0)

In [10]:
a = Value(4.0, label='a');
b = Value(3.0, label='b')
c = Value(1.0, label='c')

d = a * b; d.label='d'
e = d + c; e.label='e'

e.backward()

In [11]:

print(e.grad)
print(d.grad)
print(c.grad)
print(b.grad)
print(a.grad)

1.0
1.0
1.0
4.0
3.0
